# ALTE Common Corpus SIG pipeline runner

This notebook runs the numbered scripts. LLM outputs are provisional Tier 4 candidate material for expert review.

In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path('/content/ALTE-Common-Corpus-SIG')
DRIVE_ROOT = Path('/content/drive/MyDrive/ALTE-Common-Corpus-SIG')
LANGUAGES = ['en', 'fr', 'es', 'de', 'cs']

print('REPO_ROOT:', REPO_ROOT)
print('DRIVE_ROOT:', DRIVE_ROOT)

## Clone repository and install requirements

In [ ]:
if not REPO_ROOT.exists():
    !git clone https://github.com/Pertam/ALTE-Common-Corpus-SIG.git /content/ALTE-Common-Corpus-SIG

%cd /content/ALTE-Common-Corpus-SIG
!pip install -q -r requirements.txt || true
!pip install -q -r requirements_stage1_5.txt || true

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## Validate taxonomy

In [ ]:
!python scripts/00_validate_inputs.py \
  --taxonomy taxonomy/cefr_function_taxonomy_v0_2.csv

## Stage 01: prepare raw Leipzig sentences

In [ ]:
for lang in LANGUAGES:
    raw_path = DRIVE_ROOT / 'data' / 'raw' / lang / f'{lang}_sentences.txt'
    out_path = DRIVE_ROOT / 'data' / 'interim' / f'{lang}_sentences.parquet'
    if raw_path.exists():
        !python scripts/01_prepare_leipzig_sentences.py --lang {lang} --input {raw_path} --output {out_path}
    else:
        print('Skipping missing raw file:', raw_path)

## Stage 02: tokenise and lemmatise

In [ ]:
SPACY_MODELS = {
    'en': 'en_core_web_sm',
    'fr': 'fr_core_news_sm',
    'es': 'es_core_news_sm',
    'de': 'de_core_news_sm',
    'cs': 'cs_core_news_sm',
}

for lang in LANGUAGES:
    in_path = DRIVE_ROOT / 'data' / 'interim' / f'{lang}_sentences.parquet'
    out_dir = DRIVE_ROOT / 'data' / 'interim'
    if in_path.exists():
        !python scripts/02_tokenise_lemmatise.py --lang {lang} --model {SPACY_MODELS[lang]} --input {in_path} --out_dir {out_dir}
    else:
        print('Skipping missing prepared sentence file:', in_path)

## Stage 03: compute lemma statistics

In [ ]:
for lang in LANGUAGES:
    token_path = DRIVE_ROOT / 'data' / 'interim' / f'{lang}_tokens.parquet'
    lemma_sentence_path = DRIVE_ROOT / 'data' / 'interim' / f'{lang}_lemma_sentence_index.parquet'
    out_path = DRIVE_ROOT / 'data' / 'stage3_lemma_stats' / f'{lang}_lemma_stats.csv'
    if token_path.exists() and lemma_sentence_path.exists():
        !python scripts/03_compute_lemma_stats.py --lang {lang} --tokens {token_path} --lemma_sentence {lemma_sentence_path} --output {out_path}
    else:
        print('Skipping missing Stage 02 outputs for:', lang)

## Stage 04: sample lemmas and sentences

In [ ]:
N_LEMMAS_PER_LANGUAGE = 15
MIN_ARF_PER_MILLION = 50

for lang in LANGUAGES:
    stats_path = DRIVE_ROOT / 'data' / 'stage3_lemma_stats' / f'{lang}_lemma_stats.csv'
    lemma_sentence_path = DRIVE_ROOT / 'data' / 'interim' / f'{lang}_lemma_sentence_index.parquet'
    sentences_path = DRIVE_ROOT / 'data' / 'interim' / f'{lang}_sentences.parquet'
    out_path = DRIVE_ROOT / 'data' / 'stage4_samples' / f'stage4_{lang}_random_15_lemmas_all_sentences.csv'
    if stats_path.exists() and lemma_sentence_path.exists() and sentences_path.exists():
        !python scripts/04_sample_lemmas_and_sentences.py --lang {lang} --stats {stats_path} --lemma_sentence {lemma_sentence_path} --sentences {sentences_path} --output {out_path} --min_arf {MIN_ARF_PER_MILLION} --lemmas_n {N_LEMMAS_PER_LANGUAGE} --all_sentences
    else:
        print('Skipping missing Stage 04 inputs for:', lang)

## Stage 05/06 test run on English sample using dry_run

In [ ]:
lang = 'en'
sample_path = DRIVE_ROOT / 'data' / 'stage4_samples' / 'stage4_en_random_15_lemmas_all_sentences.csv'
outputs = DRIVE_ROOT / 'outputs'
outputs.mkdir(parents=True, exist_ok=True)

!python scripts/05a_run_pass1.py --sentences {sample_path} --taxonomy taxonomy/cefr_function_taxonomy_v0_2.csv --output {outputs / 'pass1_en.csv'} --dry_run --limit 5
!python scripts/05b_run_pass2.py --pass1 {outputs / 'pass1_en.csv'} --taxonomy taxonomy/cefr_function_taxonomy_v0_2.csv --output {outputs / 'pass2_en.csv'} --dry_run
!python scripts/05c_run_pass3.py --pass1 {outputs / 'pass1_en.csv'} --pass2 {outputs / 'pass2_en.csv'} --taxonomy taxonomy/cefr_function_taxonomy_v0_2.csv --output {outputs / 'pass3_en.csv'} --dry_run
!python scripts/06_make_final_dataset.py --samples {sample_path} --pass1 {outputs / 'pass1_en.csv'} --pass2 {outputs / 'pass2_en.csv'} --pass3 {outputs / 'pass3_en.csv'} --output {outputs / 'final_en_sentence_function_dataset.csv'}

## Stage 05 real API run

Set your API key, remove `--dry_run`, and consider starting with `--limit 10`.